# 03 Merged Strict+Cloud Feedback GA Search

Generated notebook for JOILang GA feedback experiments.

> 실행 전 `BASE_DIR`, `MODEL_KEY`, `DEVICE`, API key/env를 확인하세요.

## 0. 목적과 범위

이 노트북은 **Strict DET + Cloud Judge merged feedback**을 생성하고, 그 산출물을 기준으로 GA/advisor 실험 결과를 분석한다.

현재 repository 구조에서는:
1. `run_eval_pipeline_check.sh`가 strict DET, cloud judge, merge adapter, `advisor_rich_feedback.json` schema check를 수행한다.
2. `run_ga_search.py`는 GA fitness를 Strict DET 기반으로 계산하고, real cloud advisor 옵션으로 mutation proposal을 받을 수 있다.
3. 따라서 이 노트북은 `merged_feedback/advisor_rich_feedback.json`을 실험 전/후 evidence artifact로 보존하고, GA run의 advisor transport/effectiveness를 함께 분석한다.

In [ ]:
import os
import sys
import json
import csv
import shlex
import time
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 240)

BASE_DIR = Path(os.environ.get("JOILANG_BASE_DIR", "/root/llm/JOILang-Server")).expanduser().resolve()
VERSION_ROOT = BASE_DIR / "gpt_mg" / "version0_15_update20260413"
GA_SCRIPT = VERSION_ROOT / "scripts" / "run_ga_search.py"
RUN_BENCHMARK = VERSION_ROOT / "scripts" / "run_benchmark.py"
EVAL_PIPELINE = BASE_DIR / "run_eval_pipeline_check.sh"

DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"
DEFAULT_GENOME = VERSION_ROOT / "genomes" / "example_genome.json"

PYTHON = os.environ.get("JOI_V15_PYTHON", sys.executable)
WORKER_PYTHON = os.environ.get("JOI_V15_WORKER_PYTHON", PYTHON)

MODEL_KEY = os.environ.get("MODEL_KEY", "qwen25_coder_14b")
DEVICE = os.environ.get("JOI_V15_LOCAL_DEVICE", "cuda:0")
LOCAL_MODELS_BASE = Path(os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR", str(BASE_DIR.parent / "local_models"))).expanduser()
LOCAL_MODEL_DIR = Path(os.environ.get("JOI_V15_LOCAL_MODEL_NAME", str(LOCAL_MODELS_BASE / MODEL_KEY))).expanduser()

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NOTEBOOK_RUN_ROOT = BASE_DIR / "artifacts" / "notebook_ga_runs" / RUN_TAG
NOTEBOOK_RUN_ROOT.mkdir(parents=True, exist_ok=True)

ENV = os.environ.copy()
ENV.update({
    "JOI_V15_PYTHON": PYTHON,
    "JOI_V15_WORKER_PYTHON": WORKER_PYTHON,
    "JOI_V15_LOCAL_MODEL_BASE_DIR": str(LOCAL_MODELS_BASE),
    "JOI_V15_LOCAL_MODEL_NAME": str(LOCAL_MODEL_DIR),
    "JOI_V15_LOCAL_FILES_ONLY": os.environ.get("JOI_V15_LOCAL_FILES_ONLY", "true"),
    "JOI_V15_LOCAL_DEVICE": DEVICE,
    "JOI_V15_LOCAL_DTYPE": os.environ.get("JOI_V15_LOCAL_DTYPE", "bf16"),
    "JOI_V15_LOCAL_LOAD_IN_4BIT": os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "false"),
    "JOI_V15_LOCAL_TRUST_REMOTE_CODE": os.environ.get("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true"),
    "TRANSFORMERS_VERBOSITY": "error",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONFAULTHANDLER": "1",
})

print("BASE_DIR:", BASE_DIR)
print("VERSION_ROOT:", VERSION_ROOT)
print("GA_SCRIPT:", GA_SCRIPT, GA_SCRIPT.exists())
print("DATASET:", DATASET, DATASET.exists())
print("SERVICE_SCHEMA:", SERVICE_SCHEMA, SERVICE_SCHEMA.exists())
print("MODEL_KEY:", MODEL_KEY)
print("LOCAL_MODEL_DIR:", LOCAL_MODEL_DIR, LOCAL_MODEL_DIR.exists())
print("DEVICE:", DEVICE)
print("NOTEBOOK_RUN_ROOT:", NOTEBOOK_RUN_ROOT)

assert BASE_DIR.exists(), BASE_DIR
assert GA_SCRIPT.exists(), GA_SCRIPT
assert DATASET.exists(), DATASET
assert SERVICE_SCHEMA.exists(), SERVICE_SCHEMA

def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_cmd(cmd, *, cwd=BASE_DIR, env=ENV, log_path=None, check=True):
    """Run a shell command list, stream output, and optionally tee to a log file."""
    cmd = [str(x) for x in cmd]
    print("\n[CMD]")
    print(" ".join(shlex.quote(x) for x in cmd))
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[LOG]", log_path)
    proc = subprocess.Popen(
        cmd, cwd=str(cwd), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    lines = []
    with (open(log_path, "w", encoding="utf-8") if log_path else open(os.devnull, "w", encoding="utf-8")) as lf:
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
            if log_path:
                lf.write(line)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed rc={rc}: {' '.join(cmd)}")
    return rc, "".join(lines)

def ga_common_args():
    return [
        PYTHON, "-u", str(GA_SCRIPT),
        "--profile", "version0_15",
        "--genome-json", str(DEFAULT_GENOME),
        "--dataset", str(DATASET),
        "--service-schema", str(SERVICE_SCHEMA),
        "--model-key", MODEL_KEY,
        "--llm-mode", "worker",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",
        "--feedback-guided-mutation",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",
        "--full-run",
        "--force",
        "--progress", "verbose",
        "--retries", "0",
        "--target-detpass", "90",
    ]

def run_ga(label, scope_args, tuning_args=None, extra_args=None, output_root=None, check=True):
    output_root = Path(output_root or (NOTEBOOK_RUN_ROOT / label)).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    log_path = output_root / f"{label}.log"
    cmd = ga_common_args()
    cmd += list(scope_args)
    cmd += list(tuning_args or [])
    cmd += ["--output-root", str(output_root)]
    cmd += list(extra_args or [])
    rc, output = run_cmd(cmd, log_path=log_path, check=check)
    return output_root

def load_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def read_csv_if_exists(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)

def latest_file(root, pattern):
    files = sorted(Path(root).glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None

def summarize_ga_run(run_dir):
    run_dir = Path(run_dir)
    summary = load_json(run_dir / "ga_summary.json")
    best = load_json(run_dir / "best_genome.json")
    print("RUN_DIR:", run_dir)
    print("best_DETPass:", summary.get("best_DETPass") or summary.get("accepted_best_DETPass"))
    print("best_avg_DET:", summary.get("best_avg_DET") or summary.get("accepted_best_avg_DET"))
    print("best_genome_id:", best.get("id") or best.get("genome_id"))
    print("stop_reason:", summary.get("stop_reason"))
    for name in ["ga_summary.json", "best_genome.json", "ga_block_diffs.jsonl", "advisor_mutation_summary.csv"]:
        p = run_dir / name
        print(f"{name}:", p.exists(), p)
    return summary, best

def collect_candidate_tables(run_dir):
    cand_dir = Path(run_dir) / "candidates"
    dfs = []
    for p in sorted(cand_dir.glob("*.csv")):
        try:
            df = pd.read_csv(p)
            df["source_file"] = str(p)
            dfs.append(df)
        except Exception as e:
            print("failed:", p, e)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def collect_run_table(run_dirs):
    rows = []
    for rd in map(Path, run_dirs):
        s = load_json(rd / "ga_summary.json")
        b = load_json(rd / "best_genome.json")
        rows.append({
            "run_dir": str(rd),
            "label": rd.name,
            "best_DETPass": s.get("best_DETPass") or s.get("accepted_best_DETPass"),
            "best_avg_DET": s.get("best_avg_DET") or s.get("accepted_best_avg_DET"),
            "best_prompt_tokens": s.get("best_avg_prompt_tokens") or s.get("accepted_best_avg_prompt_tokens"),
            "stop_reason": s.get("stop_reason"),
            "best_genome_id": b.get("id") or b.get("genome_id"),
        })
    return pd.DataFrame(rows)

def generation_history(run_dir):
    s = load_json(Path(run_dir) / "ga_summary.json")
    hist = s.get("best_history") or s.get("generation_history") or []
    if not hist:
        return pd.DataFrame()
    df = pd.DataFrame(hist)
    df["run_dir"] = str(run_dir)
    return df

def plot_generation_history(run_dirs):
    hdfs = [generation_history(rd) for rd in run_dirs]
    hdfs = [df for df in hdfs if not df.empty]
    if not hdfs:
        print("No generation history found.")
        return pd.DataFrame()
    hist = pd.concat(hdfs, ignore_index=True)
    display(hist.head())

    gen_col = "generation" if "generation" in hist.columns else hist.columns[0]
    det_col = "train_det_pass_rate" if "train_det_pass_rate" in hist.columns else ("DETPass" if "DETPass" in hist.columns else None)
    avg_col = "avg_det_score" if "avg_det_score" in hist.columns else ("avg_DET" if "avg_DET" in hist.columns else None)
    tok_col = "avg_prompt_tokens" if "avg_prompt_tokens" in hist.columns else None

    if det_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[det_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel("DETPass / pass rate")
        plt.title("GA DETPass by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    if avg_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[avg_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel("Average DET")
        plt.title("GA average DET by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    if tok_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[tok_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel("Avg prompt tokens")
        plt.title("Prompt-token compression by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    return hist

def inspect_failures(run_dir, max_rows=30):
    df = collect_candidate_tables(run_dir)
    if df.empty:
        print("No candidate CSV rows found.")
        return df
    cols = [c for c in df.columns if any(k in c.lower() for k in ["row", "category", "det", "pass", "failure", "error", "prompt", "token", "candidate", "genome"])]
    display(df[cols].head(max_rows))
    return df

def compare_two_runs(run_a, run_b):
    a = collect_candidate_tables(run_a)
    b = collect_candidate_tables(run_b)
    if a.empty or b.empty:
        print("candidate table missing")
        return pd.DataFrame()
    # Use best effort join keys
    key_candidates = ["row_no", "row_id", "dataset_row", "index"]
    key = next((k for k in key_candidates if k in a.columns and k in b.columns), None)
    if key is None:
        print("No common row key. Showing summaries only.")
        display(pd.DataFrame([summarize_ga_run(run_a)[0], summarize_ga_run(run_b)[0]]))
        return pd.DataFrame()
    pass_cols = [c for c in a.columns if "pass" in c.lower() or "det" in c.lower()]
    a_small = a[[key] + pass_cols].copy()
    b_small = b[[key] + [c for c in b.columns if "pass" in c.lower() or "det" in c.lower()]].copy()
    merged = a_small.merge(b_small, on=key, suffixes=("_a", "_b"))
    display(merged.head(50))
    return merged


OPENAI_KEY = (
    os.environ.get("OPENAI_API_KEY_PROJ_BENCH")
    or os.environ.get("JOI_EVAL_OPENAI_API_KEY")
    or os.environ.get("JOI_V15_OPENAI_API_KEY")
    or os.environ.get("OPENAI_API_KEY")
)
if OPENAI_KEY:
    ENV["OPENAI_API_KEY"] = OPENAI_KEY
    ENV["OPENAI_API_KEY_PROJ_BENCH"] = OPENAI_KEY
    ENV["JOI_EVAL_OPENAI_API_KEY"] = OPENAI_KEY
ENV["LANGSMITH_TRACING"] = ENV.get("LANGSMITH_TRACING", "false")
ENV["LANGCHAIN_TRACING_V2"] = ENV.get("LANGCHAIN_TRACING_V2", "false")
print("OpenAI key configured:", bool(OPENAI_KEY))

## 1. Eval pipeline으로 merged feedback 생성

In [ ]:
def run_eval_pipeline(mode="smoke2", label=None):
    label = label or f"merged_eval_{mode}_{ts()}"
    out_log = NOTEBOOK_RUN_ROOT / f"{label}.log"
    env = ENV.copy()
    env["RUN_CLOUD"] = "1"
    cmd = [str(EVAL_PIPELINE), mode, str(BASE_DIR), DEVICE]
    rc, out = run_cmd(cmd, env=env, log_path=out_log, check=False)
    return rc, out_log

def locate_latest_eval_pipeline_artifacts():
    roots = sorted((BASE_DIR / "artifacts").glob("eval_pipeline_checks_*"), key=lambda p: p.stat().st_mtime)
    if not roots:
        return None
    root = roots[-1]
    return {
        "root": root,
        "strict_dir": root / "strict_det",
        "cloud_dir": root / "cloud_judge",
        "merge_dir": root / "merged_feedback",
        "advisor_rich_feedback": root / "merged_feedback" / "advisor_rich_feedback.json",
        "summary": root / "check_summary.tsv",
    }

RUN_MERGED_PREFLIGHT = False  # smoke2부터 시작하려면 True
if RUN_MERGED_PREFLIGHT:
    rc, eval_log = run_eval_pipeline(mode="smoke2", label=f"merged_eval_smoke2_{ts()}")
    print("rc:", rc, "log:", eval_log)

eval_artifacts = locate_latest_eval_pipeline_artifacts()
eval_artifacts

## 2. advisor_rich_feedback 구조 분석

In [ ]:
def inspect_advisor_rich_feedback(path):
    path = Path(path)
    if not path.exists():
        print("missing:", path)
        return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    print("path:", path)
    if isinstance(data, dict):
        print("top-level keys:", sorted(data.keys()))
        for key in ["metadata", "root_cause_summary", "generation_failure_summary", "evidence_quality_summary", "rows"]:
            val = data.get(key)
            if isinstance(val, list):
                print(key, "len=", len(val))
            elif isinstance(val, dict):
                print(key, "keys=", sorted(val.keys())[:30])
            else:
                print(key, type(val).__name__, str(val)[:200])
    return data

if eval_artifacts and eval_artifacts["advisor_rich_feedback"].exists():
    arf = inspect_advisor_rich_feedback(eval_artifacts["advisor_rich_feedback"])
else:
    print("No advisor_rich_feedback.json found yet. Set RUN_MERGED_PREFLIGHT=True and run previous cell.")

## 3. Row 1개 merged-context GA/advisor 실행

In [ ]:
ROW_NO = 1
MERGED_ADVISOR_ARGS = [
    "--llm-mutation-advisor",
    "--advisor-model-key", "gpt41_mini",
    "--advisor-llm-mode", "openai",
    "--advisor-trigger-mode", "always",
    "--advisor-min-population-for-child", "4",
    "--advisor-force-child-quota",
    "--advisor-compression-child-quota", "1",
    "--advisor-prefer-compression-after-detpass", "90",
    "--advisor-temperature", "0.0",
]

ROW_TUNING = [
    "--population", "4",
    "--gens", "2",
    "--min-generations", "2",
    "--max-generations", "2",
    "--sample-size", "1",
    "--validation-size", "1",
    "--cheap-eval-limit", "1",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "2400",
]

merged_row1_run = run_ga(
    label=f"merged_row{ROW_NO:03d}_g2_{ts()}",
    scope_args=["--start-row", str(ROW_NO), "--end-row", str(ROW_NO)],
    tuning_args=ROW_TUNING,
    extra_args=MERGED_ADVISOR_ARGS,
)
summarize_ga_run(merged_row1_run)

## 4. Merged feedback + advisor effectiveness 통합 분석

In [ ]:
def advisor_effectiveness_report(run_dir):
    run_dir = Path(run_dir)
    summary = load_json(run_dir / "ga_summary.json")
    advisor_csv = read_csv_if_exists(run_dir / "advisor_mutation_summary.csv")
    diffs = run_dir / "ga_block_diffs.jsonl"
    report = {
        "advisor_used": summary.get("advisor_used"),
        "advisor_proposals_generated": summary.get("advisor_proposals_generated"),
        "advisor_proposals_accepted_applied": summary.get("advisor_proposals_accepted_applied"),
        "advisor_children_scheduled": summary.get("advisor_children_scheduled"),
        "advisor_compression_children_scheduled": summary.get("advisor_compression_children_scheduled"),
        "advisor_csv_rows": len(advisor_csv),
        "accepted_rows": 0,
        "advisor_backed_diff_count": 0,
    }
    if not advisor_csv.empty and "accepted" in advisor_csv:
        report["accepted_rows"] = int(advisor_csv["accepted"].astype(str).str.lower().isin(["true", "1", "yes"]).sum())
    if diffs.exists():
        report["advisor_backed_diff_count"] = sum(
            1 for line in diffs.read_text(encoding="utf-8").splitlines()
            if "llm_advised" in line or "advisor_proposal_id" in line or "advisor_batch_id" in line
        )
    display(pd.DataFrame([report]))
    if not advisor_csv.empty:
        display(advisor_csv)
    return report

inspect_failures(merged_row1_run, max_rows=50)
advisor_effectiveness_report(merged_row1_run)

if eval_artifacts and eval_artifacts["summary"].exists():
    print("\nEval pipeline summary:")
    print(eval_artifacts["summary"].read_text(encoding="utf-8")[:4000])

## 5. Category / Full merged 실험

In [ ]:
RUN_MERGED_CATEGORY_SWEEP = False
CATEGORY_TUNING = [
    "--population", "6",
    "--gens", "3",
    "--min-generations", "2",
    "--max-generations", "3",
    "--sample-size", "4",
    "--validation-size", "4",
    "--cheap-eval-limit", "2",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "3600",
    "--limit-per-category", "5",
]
merged_category_runs = []
if RUN_MERGED_CATEGORY_SWEEP:
    for cat in range(1, 9):
        # 각 category별로 eval pipeline full을 다시 돌리지는 않고, GA scope만 category로 제한한다.
        rd = run_ga(
            label=f"merged_category{cat}_g3_{ts()}",
            scope_args=["--category", str(cat)],
            tuning_args=CATEGORY_TUNING,
            extra_args=MERGED_ADVISOR_ARGS,
        )
        merged_category_runs.append(rd)
        summarize_ga_run(rd)
        advisor_effectiveness_report(rd)

RUN_MERGED_FULL_EVAL_AND_GA = False
FULL_TUNING = [
    "--population", "16",
    "--gens", "10",
    "--min-generations", "5",
    "--max-generations", "10",
    "--sample-size", "40",
    "--validation-size", "40",
    "--cheap-eval-limit", "20",
    "--plateau-window", "3",
    "--disruptive-max-attempts", "3",
    "--timeout-sec", "7200",
]
merged_full_run = None
if RUN_MERGED_FULL_EVAL_AND_GA:
    rc, eval_log = run_eval_pipeline(mode="full", label=f"merged_eval_full_{ts()}")
    print("eval rc:", rc, "log:", eval_log)
    eval_artifacts = locate_latest_eval_pipeline_artifacts()
    print("eval artifacts:", eval_artifacts)
    merged_full_run = run_ga(
        label=f"merged_full280_g10_{ts()}",
        scope_args=["--category", "1", "--category", "2", "--category", "3", "--category", "4",
                    "--category", "5", "--category", "6", "--category", "7", "--category", "8"],
        tuning_args=FULL_TUNING,
        extra_args=MERGED_ADVISOR_ARGS,
    )
    summarize_ga_run(merged_full_run)
    advisor_effectiveness_report(merged_full_run)

## 6. 최종 그래프

In [ ]:
analysis_runs = [p for p in [locals().get("merged_row1_run"), locals().get("merged_full_run")] if p]
analysis_runs += merged_category_runs if "merged_category_runs" in globals() else []
analysis_runs = [Path(p) for p in analysis_runs if p]

display(collect_run_table(analysis_runs))
plot_generation_history(analysis_runs)